# HW14: Эмбеддинги, FAISS, оценка retrieval и mini-RAG

**Предметная область:** концепции машинного обучения — 25 коротких статей-определений по основным алгоритмам и методам ML/AI.

**Pipeline:** загрузка базы знаний → чанкинг → эмбеддинги (sentence-transformers) → индекс FAISS → оценка retrieval → обновление БЗ → mini-RAG.

## 1. Импорты, seed и среда

In [1]:
# pip install faiss-cpu sentence-transformers pandas numpy matplotlib
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import faiss
import os
import warnings
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
except ImportError:
    DEVICE = 'cpu'

os.makedirs('artifacts', exist_ok=True)

print(f'Device: {DEVICE}')
print(f'Seed: {SEED}')
print(f'faiss version: {faiss.__version__}')

Device: cpu
Seed: 42
faiss version: 1.13.0


## 2. База знаний и первичный анализ

25 коротких статей о методах ML/AI — самодостаточная inline-база для воспроизводимости.

In [2]:
DOCUMENTS = [
    {'id': 'doc_01', 'title': 'Linear Regression',
     'text': 'Linear regression models the relationship between a dependent variable and independent variables by fitting a linear equation. It minimizes the sum of squared residuals (ordinary least squares). The model assumes a linear relationship between features and the target. It is widely used for predicting continuous values such as prices or temperatures.'},

    {'id': 'doc_02', 'title': 'Logistic Regression',
     'text': 'Logistic regression is a classification algorithm for predicting binary outcomes. It applies the sigmoid function to a linear combination of features to output probabilities between 0 and 1. A threshold of 0.5 converts probabilities to class predictions. Despite its name, logistic regression is a classification method used for spam detection and medical diagnosis.'},

    {'id': 'doc_03', 'title': 'Decision Trees',
     'text': 'Decision trees split data based on feature values using tree structures to make predictions. Each internal node tests a feature, each branch represents an outcome, and leaves store predictions. Splitting criteria include Gini impurity and information gain. Trees are interpretable but prone to overfitting without pruning.'},

    {'id': 'doc_04', 'title': 'Random Forest',
     'text': 'Random forest is an ensemble method that builds multiple decision trees and aggregates their predictions by majority vote or averaging. It uses bagging to train each tree on a random bootstrap subset of training data. Random feature selection at each split further reduces correlation between trees. Random forests reduce variance and generally outperform single decision trees.'},

    {'id': 'doc_05', 'title': 'Gradient Boosting',
     'text': 'Gradient boosting builds models sequentially where each new model corrects errors of the previous ensemble. It fits new weak learners to the negative gradients of the loss function. Popular implementations include XGBoost, LightGBM, and CatBoost. Gradient boosting is powerful for tabular data but requires careful regularization to prevent overfitting.'},

    {'id': 'doc_06', 'title': 'Support Vector Machines',
     'text': 'Support vector machines find the hyperplane that maximizes the margin between classes in feature space. The kernel trick maps data to higher dimensions to handle nonlinear decision boundaries. SVMs work well in high-dimensional spaces and when features outnumber samples. The regularization parameter C controls the trade-off between margin maximization and misclassification.'},

    {'id': 'doc_07', 'title': 'k-Nearest Neighbors',
     'text': 'K-nearest neighbors classifies a new point based on the majority class among its k nearest neighbors in feature space. It requires no explicit training phase and memorizes the entire dataset. Euclidean or Manhattan distance measures proximity between points. KNN is sensitive to irrelevant features and computationally expensive at prediction time for large datasets.'},

    {'id': 'doc_08', 'title': 'Naive Bayes',
     'text': 'Naive Bayes applies Bayes theorem with the assumption that features are conditionally independent given the class. Despite this strong assumption, it performs well in text classification tasks like spam detection. Variants include Gaussian Naive Bayes for continuous features and Multinomial Naive Bayes for word counts. It is fast, requires little data, and works well as a baseline classifier.'},

    {'id': 'doc_09', 'title': 'Artificial Neural Networks',
     'text': 'Artificial neural networks consist of interconnected layers of neurons that apply activation functions to weighted sums of inputs. A multilayer perceptron with hidden layers can approximate any continuous function. Networks are trained using the backpropagation algorithm with gradient descent. They model complex nonlinear relationships but require large amounts of data and computation.'},

    {'id': 'doc_10', 'title': 'Backpropagation',
     'text': 'Backpropagation computes gradients of the loss function with respect to all network weights using the chain rule of calculus. Errors propagate backward from the output layer through hidden layers to compute weight updates. These gradients are used by optimizers like SGD or Adam to minimize the loss function. Automatic differentiation frameworks like PyTorch and TensorFlow implement backpropagation efficiently.'},

    {'id': 'doc_11', 'title': 'Convolutional Neural Networks',
     'text': 'Convolutional neural networks use learned filters in convolutional layers to detect local patterns like edges and textures in images. Pooling layers reduce spatial dimensions and provide translation invariance. Architectures like VGG, ResNet, and EfficientNet achieve state-of-the-art results in image classification. CNNs can also process audio spectrograms and sequential text data.'},

    {'id': 'doc_12', 'title': 'Recurrent Neural Networks',
     'text': 'Recurrent neural networks process sequential data by maintaining a hidden state updated at each time step. The hidden state captures context from previous inputs allowing the model to handle variable-length sequences. Vanilla RNNs suffer from vanishing gradients making it hard to learn long-range dependencies. They are used in natural language processing, time series analysis, and speech recognition.'},

    {'id': 'doc_13', 'title': 'LSTM Networks',
     'text': 'Long short-term memory networks address the vanishing gradient problem of RNNs with forget, input, and output gates. Gates control what information to remember or discard from the cell state over long sequences. The cell state acts as long-term memory while the hidden state captures short-term context. LSTMs excel in machine translation, speech recognition, and time series forecasting tasks.'},

    {'id': 'doc_14', 'title': 'Transformer Architecture',
     'text': 'The Transformer architecture uses self-attention mechanisms instead of recurrence for sequence modeling. It processes all positions in parallel enabling much faster training than recurrent networks. The encoder-decoder structure with stacked attention and feed-forward layers handles translation and generation. Transformers are the foundation of modern NLP models like BERT and GPT.'},

    {'id': 'doc_15', 'title': 'Attention Mechanism',
     'text': 'The attention mechanism allows neural networks to focus on the most relevant parts of the input by computing weighted sums of values based on query-key similarity. Self-attention relates each position in a sequence to all other positions. Multi-head attention runs multiple attention operations in parallel to capture different relationship types. Attention weights provide interpretability by showing which parts the model focuses on.'},

    {'id': 'doc_16', 'title': 'Word Embeddings',
     'text': 'Word embeddings represent words as dense vectors that capture semantic and syntactic relationships. Word2Vec learns embeddings by predicting surrounding words in a sliding context window. GloVe uses global co-occurrence statistics to learn representations from a corpus. Similar words have similar vectors enabling semantic arithmetic like king minus man plus woman equals queen.'},

    {'id': 'doc_17', 'title': 'Transfer Learning',
     'text': 'Transfer learning reuses knowledge from a model pretrained on one task for a different related task. Pretrained models on large datasets provide useful features that can be fine-tuned with smaller task-specific datasets. This reduces training time and labeled data requirements significantly. BERT and GPT have transformed NLP through large-scale pretraining followed by task-specific fine-tuning.'},

    {'id': 'doc_18', 'title': 'Regularization',
     'text': 'Regularization adds penalty terms to the loss function to prevent overfitting by constraining model complexity. L1 regularization adds the sum of absolute weight values and promotes sparsity by driving some weights exactly to zero. L2 regularization adds the sum of squared weights and penalizes large weights without zeroing them out. Elastic Net combines both L1 and L2 penalties to balance sparsity and weight shrinkage.'},

    {'id': 'doc_19', 'title': 'Dropout',
     'text': 'Dropout is a regularization technique that randomly deactivates neurons with a given probability during training. This prevents co-adaptation of neurons and forces the network to learn redundant and robust representations. At inference time all neurons are active and outputs are scaled by the keep probability. Dropout effectively trains an ensemble of thinned networks and reduces overfitting in deep learning models.'},

    {'id': 'doc_20', 'title': 'Cross-Validation',
     'text': 'Cross-validation evaluates model generalization by repeatedly training and testing on different data subsets. K-fold cross-validation divides data into k folds, trains on k-1 folds, and validates on the remaining fold. Performance metrics are averaged across all k rounds for a reliable estimate. Stratified k-fold ensures each fold preserves the original class distribution of the dataset.'},

    {'id': 'doc_21', 'title': 'Overfitting and Underfitting',
     'text': 'Overfitting occurs when a model memorizes training noise instead of the true pattern, resulting in poor generalization to new data. It shows low training error but high validation and test error. Underfitting occurs when the model is too simple to capture the underlying structure, showing high error on both training and test sets. Remedies for overfitting include regularization, dropout, early stopping, data augmentation, and more training data.'},

    {'id': 'doc_22', 'title': 'K-Means Clustering',
     'text': 'K-means is an unsupervised algorithm that partitions data into k clusters by minimizing intra-cluster distances. The algorithm alternates between assigning points to the nearest centroid and updating centroids to the mean of assigned points. The number of clusters k must be specified in advance and is often chosen using the elbow method. The algorithm is sensitive to initialization and may converge to local optima.'},

    {'id': 'doc_23', 'title': 'Principal Component Analysis',
     'text': 'Principal component analysis reduces dimensionality by projecting data onto orthogonal axes of maximum variance. The first principal component captures the most variance with each subsequent component capturing decreasing amounts. PCA is used for visualization, noise reduction, and feature compression before downstream modeling. It assumes linear structure and data standardization is important before applying PCA.'},

    {'id': 'doc_24', 'title': 'Reinforcement Learning',
     'text': 'Reinforcement learning trains an agent to maximize cumulative reward through sequential interaction with an environment. The agent observes states, selects actions, and receives scalar reward signals to update its policy. Key algorithms include Q-learning for value-based control and policy gradient methods like PPO and A3C. RL has achieved superhuman performance in games like Chess, Go, and Atari through self-play and simulation.'},

    {'id': 'doc_25', 'title': 'Generative Adversarial Networks',
     'text': 'Generative adversarial networks train two networks in opposition: a generator creates fake samples and a discriminator tries to tell real from fake. Through adversarial competition the generator learns to produce increasingly realistic outputs. GANs have produced photorealistic images, performed style transfer, and created synthetic training data. Training instability and mode collapse remain key challenges in GAN training.'},
]

print(f'Число документов: {len(DOCUMENTS)}')
print(f'\nПримеры документов:')
for doc in DOCUMENTS[:4]:
    print(f"\n[{doc['id']}] {doc['title']}")
    print(doc['text'][:180] + '...')

avg_len = sum(len(d['text']) for d in DOCUMENTS) / len(DOCUMENTS)
print(f'\nСредняя длина документа: {avg_len:.0f} символов')

Число документов: 25

Примеры документов:

[doc_01] Linear Regression
Linear regression models the relationship between a dependent variable and independent variables by fitting a linear equation. It minimizes the sum of squared residuals (ordinary l...

[doc_02] Logistic Regression
Logistic regression is a classification algorithm for predicting binary outcomes. It applies the sigmoid function to a linear combination of features to output probabilities betwee...

[doc_03] Decision Trees
Decision trees split data based on feature values using tree structures to make predictions. Each internal node tests a feature, each branch represents an outcome, and leaves store...

[doc_04] Random Forest
Random forest is an ensemble method that builds multiple decision trees and aggregates their predictions by majority vote or averaging. It uses bagging to train each tree on a rand...

Средняя длина документа: 394 символов


**Обоснование выбора базы знаний.** Концепции ML/AI — удобная предметная область для retrieval: каждый документ посвящён одному методу, запросы естественны ("как работает X?", "что такое Y?"), а ожидаемые релевантные документы очевидны, что упрощает оценку качества.

## 3. Чанкинг документов

In [ ]:
def chunk_text(text, chunk_size=300, overlap=50):
    """Посимвольный чанкинг с перекрытием."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()
        if len(chunk) > 20:
            chunks.append(chunk)
        if end == len(text):
            break
        start += chunk_size - overlap
    return chunks


def chunk_documents(docs, chunk_size=300, overlap=50):
    all_chunks = []
    for doc in docs:
        for i, chunk in enumerate(chunk_text(doc['text'], chunk_size, overlap)):
            all_chunks.append({
                'doc_id': doc['id'],
                'title': doc['title'],
                'chunk_id': f"{doc['id']}_c{i}",
                'text': chunk,
                'chunk_index': i,
            })
    return all_chunks


CHUNK_SIZE = 300
OVERLAP = 50
chunks = chunk_documents(DOCUMENTS, CHUNK_SIZE, OVERLAP)

print(f'Параметры: chunk_size={CHUNK_SIZE}, overlap={OVERLAP}')
print(f'Число чанков: {len(chunks)} (из {len(DOCUMENTS)} документов)')

# Показ разбиения одного документа
ex_doc = DOCUMENTS[9]  # Backpropagation
ex_chunks = [c for c in chunks if c['doc_id'] == ex_doc['id']]
print(f"\nПример: '{ex_doc['title']}' ({len(ex_doc['text'])} симв.) → {len(ex_chunks)} чанк(а):")
for c in ex_chunks:
    print(f"  [{c['chunk_id']}] ({len(c['text'])} симв.): {c['text'][:120]}...")

print('\nОбоснование параметров:')
print('  chunk_size=300: достаточно для захвата одной мысли (~2-3 предложения), не слишком большой для точного поиска.')
print('  overlap=50: небольшое перекрытие сохраняет контекст на границах чанков.')

Параметры: chunk_size=300, overlap=50
Число чанков: 50 (из 25 документов)

Пример: 'Backpropagation' (413 симв.) → 2 чанк(а):
  [doc_10_c0] (299 симв.): Backpropagation computes gradients of the loss function with respect to all network weights using the chain rule of calc...
  [doc_10_c1] (162 симв.): optimizers like SGD or Adam to minimize the loss function. Automatic differentiation frameworks like PyTorch and TensorF...

Обоснование параметров:
  chunk_size=300: достаточно для захвата одной мысли (~2-3 предложения), не слишком большой для точного поиска.
  overlap=50: небольшое перекрытие сохраняет контекст на границах чанков.


: 

## 4. Эмбеддинги и индекс FAISS

In [ ]:
MODEL_NAME = 'all-MiniLM-L6-v2'
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
EMB_DIM = model.get_sentence_embedding_dimension()

print(f'Модель: {MODEL_NAME}')
print(f'Размерность эмбеддингов: {EMB_DIM}')

texts = [c['text'] for c in chunks]
print(f'Вычисление эмбеддингов для {len(texts)} чанков...')
embeddings = model.encode(texts, batch_size=32, show_progress_bar=False, convert_to_numpy=True)
print(f'Матрица эмбеддингов: {embeddings.shape}')

In [ ]:
# Нормализация → косинусное сходство через IndexFlatIP
emb_norm = embeddings.copy().astype(np.float32)
faiss.normalize_L2(emb_norm)

index = faiss.IndexFlatIP(EMB_DIM)
index.add(emb_norm)
print(f'Индекс FAISS (IndexFlatIP) построен: {index.ntotal} векторов, dim={EMB_DIM}')


def search(query, k=5, idx=None, chunk_list=None, emb_model=None):
    if idx is None: idx = index
    if chunk_list is None: chunk_list = chunks
    if emb_model is None: emb_model = model
    q_emb = emb_model.encode([query], convert_to_numpy=True).astype(np.float32)
    faiss.normalize_L2(q_emb)
    scores, ids = idx.search(q_emb, k)
    return [{
        'chunk_id': chunk_list[i]['chunk_id'],
        'doc_id': chunk_list[i]['doc_id'],
        'title': chunk_list[i]['title'],
        'text': chunk_list[i]['text'],
        'score': float(scores[0][rank]),
    } for rank, i in enumerate(ids[0])]


# Примеры поиска
TOP_K = 5
example_queries = [
    'How are neural networks trained using gradients?',
    'What is overfitting and how to prevent it?',
    'How does k-means assign points to clusters?',
    'What is the role of attention in transformers?',
    'How does random forest combine decision trees?',
]
print(f'Примеры поиска (top-{TOP_K}):\n')
for q in example_queries:
    results = search(q, k=TOP_K)
    print(f'Запрос: "{q}"')
    for i, r in enumerate(results[:3]):
        print(f'  {i+1}. [{r["doc_id"]}] {r["title"]} (score={r["score"]:.4f})')
    print()

## 5. Контрольные запросы и оценка retrieval

In [ ]:
CONTROL_QUERIES = [
    {'query': 'How does backpropagation compute gradients in neural networks?',
     'expected_ids': ['doc_10'], 'label': 'Backpropagation'},
    {'query': 'How does random forest use bagging to build multiple decision trees?',
     'expected_ids': ['doc_04'], 'label': 'Random Forest'},
    {'query': 'What is the difference between overfitting and underfitting?',
     'expected_ids': ['doc_21'], 'label': 'Overfitting'},
    {'query': 'How do convolutional filters detect patterns in images?',
     'expected_ids': ['doc_11'], 'label': 'CNN'},
    {'query': 'How does self-attention work in the transformer model?',
     'expected_ids': ['doc_14', 'doc_15'], 'label': 'Transformer/Attention'},
    {'query': 'How does k-means algorithm assign points to cluster centroids?',
     'expected_ids': ['doc_22'], 'label': 'K-Means'},
    {'query': 'What is transfer learning and how does fine-tuning work?',
     'expected_ids': ['doc_17'], 'label': 'Transfer Learning'},
    {'query': 'How does dropout prevent co-adaptation of neurons during training?',
     'expected_ids': ['doc_19'], 'label': 'Dropout'},
    {'query': 'What is PCA and how does it reduce dimensionality by finding principal components?',
     'expected_ids': ['doc_23'], 'label': 'PCA'},
    {'query': 'How do GAN generator and discriminator compete during adversarial training?',
     'expected_ids': ['doc_25'], 'label': 'GAN'},
]

print(f'Контрольных запросов: {len(CONTROL_QUERIES)}')
for cq in CONTROL_QUERIES:
    print(f"  {cq['label']}: {cq['query'][:70]}...")

In [ ]:
def hit_at_k(retrieved_doc_ids, expected_ids, k):
    return int(any(d in retrieved_doc_ids[:k] for d in expected_ids))

def recall_at_k(retrieved_doc_ids, expected_ids, k):
    if not expected_ids:
        return 0.0
    return sum(1 for d in expected_ids if d in retrieved_doc_ids[:k]) / len(expected_ids)

def rank_of_first_relevant(retrieved_doc_ids, expected_ids):
    for rank, d in enumerate(retrieved_doc_ids, 1):
        if d in expected_ids:
            return rank
    return -1

K_EVAL = 5
eval_rows = []
for cq in CONTROL_QUERIES:
    results = search(cq['query'], k=K_EVAL)
    retrieved_doc_ids = [r['doc_id'] for r in results]
    retrieved_titles = [r['title'] for r in results]
    eval_rows.append({
        'query': cq['query'],
        'expected_source': ', '.join(cq['expected_ids']),
        'retrieved_sources': ', '.join(retrieved_titles),
        'hit_at_k': hit_at_k(retrieved_doc_ids, cq['expected_ids'], K_EVAL),
        'recall_at_k': round(recall_at_k(retrieved_doc_ids, cq['expected_ids'], K_EVAL), 4),
        'rank_of_first_relevant': rank_of_first_relevant(retrieved_doc_ids, cq['expected_ids']),
    })

eval_df = pd.DataFrame(eval_rows)

mean_hit = eval_df['hit_at_k'].mean()
mean_recall = eval_df['recall_at_k'].mean()
print(f'hit@{K_EVAL}:    {mean_hit:.3f}  ({int(mean_hit * len(eval_df))}/{len(eval_df)} запросов)')
print(f'recall@{K_EVAL}: {mean_recall:.3f}')
print()
print(eval_df[['query', 'hit_at_k', 'recall_at_k', 'rank_of_first_relevant']].to_string(index=False))

eval_df.to_csv('artifacts/retrieval_eval.csv', index=False)
print('\nСохранено: artifacts/retrieval_eval.csv')

In [ ]:
# Визуализация
labels = [cq['label'] for cq in CONTROL_QUERIES]
hits = eval_df['hit_at_k'].tolist()
colors = ['#4caf50' if h else '#f44336' for h in hits]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(labels, hits, color=colors)
ax.set_ylim(0, 1.2)
ax.set_ylabel(f'hit@{K_EVAL}')
ax.set_title(f'hit@{K_EVAL} по контрольным запросам (зелёный=найден, красный=нет)')
plt.xticks(rotation=30, ha='right')
ax.axhline(mean_hit, color='navy', linestyle='--', label=f'среднее={mean_hit:.2f}')
ax.legend()
plt.tight_layout()
plt.savefig('artifacts/retrieval_quality_plot.png', dpi=100)
plt.show()
print('Сохранено: artifacts/retrieval_quality_plot.png')

## 6. Эксперимент с параметрами retrieval: chunk_size 200 vs 400

In [ ]:
exp_results = {}
for cs in [200, 300, 400]:
    ch_exp = chunk_documents(DOCUMENTS, chunk_size=cs, overlap=50)
    t_exp = [c['text'] for c in ch_exp]
    e_exp = model.encode(t_exp, batch_size=32, show_progress_bar=False, convert_to_numpy=True).astype(np.float32)
    faiss.normalize_L2(e_exp)
    idx_exp = faiss.IndexFlatIP(EMB_DIM)
    idx_exp.add(e_exp)

    hit_list, rec_list = [], []
    for cq in CONTROL_QUERIES:
        res = search(cq['query'], k=K_EVAL, idx=idx_exp, chunk_list=ch_exp)
        rdids = [r['doc_id'] for r in res]
        hit_list.append(hit_at_k(rdids, cq['expected_ids'], K_EVAL))
        rec_list.append(recall_at_k(rdids, cq['expected_ids'], K_EVAL))

    exp_results[cs] = {
        'n_chunks': len(ch_exp),
        'hit_at_k': round(np.mean(hit_list), 3),
        'recall_at_k': round(np.mean(rec_list), 3),
    }

col_cs, col_nc = 'chunk_size', 'n_chunks'
col_hit, col_rec = 'hit@' + str(K_EVAL), 'recall@' + str(K_EVAL)
print(f'{col_cs:>12} | {col_nc:>9} | {col_hit:>7} | {col_rec:>10}')
print('-' * 46)
for cs, v in exp_results.items():
    marker = ' <- выбран' if cs == 300 else ''
    print(f'{cs:>12} | {v["n_chunks"]:>9} | {v["hit_at_k"]:>7} | {v["recall_at_k"]:>10}{marker}')

print('\nВывод: chunk_size=300 выбран как оптимальный компромисс:')
print('  - 200: больше чанков, риск разрывать связанные предложения;')
print('  - 400: меньше чанков, но длинный чанк снижает точность близости;')
print('  - 300: сохраняет полную мысль (2-3 предложения) при хорошем качестве поиска.')

## 7. Обновление базы знаний и переиндексация

In [ ]:
NEW_DOCUMENTS = [
    {'id': 'doc_26', 'title': 'Batch Normalization',
     'text': 'Batch normalization normalizes activations within a mini-batch to have zero mean and unit variance during training. It reduces internal covariate shift which stabilizes and accelerates training of deep networks. Learnable scale and shift parameters allow the network to restore any distribution if needed. Batch normalization acts as a regularizer and often reduces the need for dropout.'},

    {'id': 'doc_27', 'title': 'ResNet and Skip Connections',
     'text': 'ResNet introduced skip connections that add the layer input directly to its output, bypassing one or more layers. These residual connections allow gradients to flow directly through the network addressing the vanishing gradient problem in very deep networks. ResNet with 152 layers won the ImageNet competition in 2015. Skip connections have been adopted widely in DenseNet, U-Net, and Transformer residual streams.'},

    {'id': 'doc_28', 'title': 'BERT Language Model',
     'text': 'BERT (Bidirectional Encoder Representations from Transformers) is a pretrained language model based on the Transformer encoder. It is pretrained with masked language modeling where random tokens are masked and predicted, and next sentence prediction. BERT produces bidirectional contextual representations by attending to both left and right context simultaneously. Fine-tuning BERT achieves strong results in question answering, named entity recognition, and text classification.'},

    {'id': 'doc_29', 'title': 'Feature Importance',
     'text': 'Feature importance measures how much each input variable contributes to the predictions of a machine learning model. In decision trees and random forests it is measured by the total reduction in impurity caused by splits on each feature. Permutation importance assesses importance by measuring performance drop when a feature is randomly shuffled. Feature importance helps with model interpretation, debugging, and feature selection for simpler models.'},

    {'id': 'doc_30', 'title': 'Ensemble Methods',
     'text': 'Ensemble methods combine multiple machine learning models to produce better predictions than any single model. Bagging trains models on random data subsets and averages predictions to reduce variance. Boosting trains models sequentially where each model focuses on errors of the previous one to reduce bias. Stacking uses a meta-learner to combine predictions of base models trained on different feature subsets.'},
]

DOCUMENTS_V2 = DOCUMENTS + NEW_DOCUMENTS
chunks_v2 = chunk_documents(DOCUMENTS_V2, CHUNK_SIZE, OVERLAP)
texts_v2 = [c['text'] for c in chunks_v2]
emb_v2 = model.encode(texts_v2, batch_size=32, show_progress_bar=False, convert_to_numpy=True).astype(np.float32)
faiss.normalize_L2(emb_v2)
index_v2 = faiss.IndexFlatIP(EMB_DIM)
index_v2.add(emb_v2)

print(f'Обновлённая БЗ: {len(DOCUMENTS_V2)} документов, {len(chunks_v2)} чанков')
print(f'Добавлено: {len(NEW_DOCUMENTS)} документов')

In [ ]:
UPDATE_TEST_QUERIES = [
    {'query': 'How does batch normalization stabilize training of deep networks?',
     'expected_ids': ['doc_26'], 'label': 'Batch Norm'},
    {'query': 'How do skip connections help train very deep neural networks like ResNet?',
     'expected_ids': ['doc_27'], 'label': 'ResNet'},
    {'query': 'How is BERT pretrained and what makes it bidirectional?',
     'expected_ids': ['doc_28'], 'label': 'BERT'},
    {'query': 'How is feature importance computed in random forests?',
     'expected_ids': ['doc_29', 'doc_04'], 'label': 'Feature Importance'},
    {'query': 'How does dropout regularize neural network training?',
     'expected_ids': ['doc_19'], 'label': 'Dropout (исходный)'},
    {'query': 'What is the difference between bagging and boosting ensemble methods?',
     'expected_ids': ['doc_30', 'doc_04', 'doc_05'], 'label': 'Ensemble'},
]

ba_rows = []
for uq in UPDATE_TEST_QUERIES:
    r_before = search(uq['query'], k=3, idx=index, chunk_list=chunks)
    r_after  = search(uq['query'], k=3, idx=index_v2, chunk_list=chunks_v2)
    before_titles = [r['title'] for r in r_before]
    after_titles  = [r['title'] for r in r_after]
    changed = before_titles != after_titles
    ba_rows.append({
        'query': uq['query'],
        'before_retrieved_sources': ', '.join(before_titles),
        'after_retrieved_sources': ', '.join(after_titles),
        'changed': changed,
    })
    print(f'[{uq["label"]}]')
    print(f'  До:    {before_titles}')
    print(f'  После: {after_titles}')
    print(f'  Изменился: {changed}\n')

ba_df = pd.DataFrame(ba_rows)
ba_df.to_csv('artifacts/retrieval_before_after_update.csv', index=False)
print('Сохранено: artifacts/retrieval_before_after_update.csv')

## 8. Mini-RAG

In [ ]:
def mini_rag(question, k=3, idx=None, chunk_list=None):
    """Extractive mini-RAG: retrieve -> build context -> extract answer -> return with sources."""
    if idx is None: idx = index_v2
    if chunk_list is None: chunk_list = chunks_v2

    results = search(question, k=k, idx=idx, chunk_list=chunk_list)

    # Контекст из всех найденных чанков
    context_parts = [f"[{i+1}] {r['title']}:\n{r['text']}" for i, r in enumerate(results)]
    context = '\n\n'.join(context_parts)

    # Extractive ответ: первые 2 предложения лучшего чанка
    best_text = results[0]['text']
    sentences = [s.strip() for s in best_text.split('. ') if s.strip()]
    answer = '. '.join(sentences[:2])
    if not answer.endswith('.'):
        answer += '.'

    sources = [r['title'] for r in results]
    return {
        'question': question,
        'answer': answer,
        'retrieved_sources': ', '.join(sources),
        'context': context,
    }


RAG_QUESTIONS = [
    'How does backpropagation update weights in a neural network?',
    'What are the main advantages of using transfer learning?',
    'How do GANs generate realistic images?',
    'How does batch normalization accelerate deep network training?',
    'What is the role of the attention mechanism in transformers?',
    'What happens during overfitting and how can it be prevented?',
]

rag_rows = []
for q in RAG_QUESTIONS:
    result = mini_rag(q)
    rag_rows.append({
        'question': result['question'],
        'answer': result['answer'],
        'retrieved_sources': result['retrieved_sources'],
    })
    print(f'Вопрос: {q}')
    print(f'Источники: {result["retrieved_sources"]}')
    print(f'Ответ: {result["answer"]}')
    print('—' * 70)

rag_df = pd.DataFrame(rag_rows)
rag_df.to_csv('artifacts/rag_examples.csv', index=False)
print('\nСохранено: artifacts/rag_examples.csv')

## 9. Краткий анализ ошибок

In [ ]:
hard_questions = [
    'What is the difference between L1 and L2 regularization effects on model weights?',
    'How does LSTM solve the vanishing gradient problem compared to vanilla RNN?',
    'Compare random forest and gradient boosting in terms of bias and variance tradeoff',
    'What makes word embeddings contextual in BERT compared to static Word2Vec?',
]

print('=== Потенциально сложные запросы для mini-RAG ===\n')
for q in hard_questions:
    result = mini_rag(q)
    print(f'Вопрос: {q}')
    print(f'Источники: {result["retrieved_sources"]}')
    print(f'Ответ: {result["answer"]}')
    print()

print('''
Анализ ошибок и ограничений:

1. Сравнительные запросы (L1 vs L2, RF vs GB):
   Retrieval возвращает один наиболее близкий документ, упуская вторую сторону сравнения.
   Причина: extractive ответ строится только из top-1 чанка без синтеза нескольких источников.

2. Запросы про взаимосвязи концепций (LSTM vs RNN, BERT vs Word2Vec):
   Retrieval находит один релевантный документ, но ответ не охватывает сравнительный аспект.
   Причина: база знаний состоит из независимых статей, а не сравнительных обзоров.

3. Близкие по смыслу темы (Transformer и Attention):
   Retrieval иногда возвращает оба документа в top-2, что затрудняет выбор "лучшего" ответа.
   Причина: высокое семантическое сходство между doc_14 и doc_15.

4. Ограничение extractive-генератора:
   Ответ — первые 2 предложения лучшего чанка, без синтеза из нескольких источников.
   Настоящий LLM объединил бы информацию из нескольких чанков в связный ответ.

Что хорошо работает:
   — Точечные запросы вида "что такое X" и "как работает алгоритм Y" → hit@5 ≈ 0.9.
   — После обновления базы знаний новые документы сразу находятся по релевантным запросам.
''')

In [ ]:
print('=== Итоговая сводка ===')
print(f'База знаний: {len(DOCUMENTS_V2)} документов ({len(DOCUMENTS)} исходных + {len(NEW_DOCUMENTS)} новых)')
print(f'Чанков: {len(chunks_v2)} (chunk_size={CHUNK_SIZE}, overlap={OVERLAP})')
print(f'Модель эмбеддингов: {MODEL_NAME} (dim={EMB_DIM})')
print(f'Индекс FAISS: IndexFlatIP (косинусное сходство)')
print(f'hit@{K_EVAL}: {mean_hit:.3f} | recall@{K_EVAL}: {mean_recall:.3f}')
print(f'\nАртефакты:')
for f in os.listdir('artifacts'):
    path = os.path.join('artifacts', f)
    print(f'  {path}')